# Ethiopia Climate EDA
NASA POWER MERRA-2 Daily Data — Addis Ababa (2015–2026)

## 1. Data Loading & Date Parsing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/ethiopia (1).csv')
df['Country'] = 'Ethiopia'
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['Date'].dt.month
print(df.shape)
df.head()

## 2. Summary Statistics & Missing Value Report

In [ ]:
df.replace(-999, np.nan, inplace=True)

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)

**Duplicates:** No duplicate rows were found in the Ethiopia dataset, which suggests the data was cleanly extracted from NASA POWER with one row per day.

In [ ]:
df.describe()

**Interpretation of Summary Statistics:**
- T2M (mean temperature) averages around 16–18°C, consistent with Addis Ababa's high-altitude climate.
- PRECTOTCORR shows a high standard deviation relative to its mean, indicating strong seasonal variability in rainfall.
- PS (surface pressure) is consistently lower than sea-level norms (~79–81 kPa), reflecting Addis Ababa's elevation of ~2,355m.
- RH2M ranges widely, suggesting distinct wet and dry seasons.

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
print(missing_report[missing_report['Missing %'] > 0])

**Missing Values:**
Any column with >5% missing values is flagged above. Missing data in NASA POWER typically occurs when the satellite model cannot produce a reliable estimate for that day. These will be handled via forward-fill in the cleaning step below.

## 3. Outlier Detection & Basic Cleaning

In [ ]:
climate_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

z_scores = df[climate_cols].apply(lambda col: np.abs(stats.zscore(col.dropna())))
outlier_counts = (z_scores > 3).sum()
print('Outlier counts per column (|Z| > 3):')
print(outlier_counts)

**Outlier Decision:**
Outliers with |Z| > 3 are retained rather than dropped. Since this is real climate data from NASA POWER, extreme values likely represent genuine weather events (e.g. heavy rainfall days, heat spikes) rather than data errors. Dropping them would distort trend analysis and extreme event frequency counts required in Task 3.

In [ ]:
# Forward-fill missing values for weather variables
df[climate_cols] = df[climate_cols].fillna(method='ffill')

# Drop rows where more than 30% of values are still missing
threshold = len(df.columns) * 0.7
df.dropna(thresh=int(threshold), inplace=True)
print('Shape after cleaning:', df.shape)

In [ ]:
import os
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/ethiopia_clean.csv', index=False)
print('Exported to data/ethiopia_clean.csv')

## 4. Time Series Analysis

In [ ]:
monthly_temp = df.groupby('Month')['T2M'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_temp.index, monthly_temp.values, marker='o', color='tomato')

warmest = monthly_temp.idxmax()
coolest = monthly_temp.idxmin()
ax.annotate(f'Warmest\n{monthly_temp[warmest]:.1f}°C', xy=(warmest, monthly_temp[warmest]),
            xytext=(warmest + 0.3, monthly_temp[warmest] + 0.3), fontsize=9, color='red')
ax.annotate(f'Coolest\n{monthly_temp[coolest]:.1f}°C', xy=(coolest, monthly_temp[coolest]),
            xytext=(coolest + 0.3, monthly_temp[coolest] - 0.5), fontsize=9, color='blue')

ax.set_title('Monthly Average Temperature — Ethiopia (2015–2026)')
ax.set_xlabel('Month')
ax.set_ylabel('Avg T2M (°C)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()

In [ ]:
monthly_rain = df.groupby('Month')['PRECTOTCORR'].sum() / df['YEAR'].nunique()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(monthly_rain.index, monthly_rain.values, color='steelblue')

peak = monthly_rain.idxmax()
ax.annotate(f'Peak\n{monthly_rain[peak]:.1f}mm', xy=(peak, monthly_rain[peak]),
            xytext=(peak + 0.3, monthly_rain[peak] + 2), fontsize=9, color='navy')

ax.set_title('Monthly Average Total Precipitation — Ethiopia (2015–2026)')
ax.set_xlabel('Month')
ax.set_ylabel('Avg Total PRECTOTCORR (mm)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()

**Time Series Observations:**
- Temperature peaks around March–May (dry season) and dips during July–August (main rainy season), consistent with Ethiopia's highland climate.
- Precipitation peaks in July–August, corresponding to the Kiremt (main rainy season). A secondary smaller peak may appear around April (Belg season).
- The inverse relationship between temperature and rainfall is a classic signature of monsoon-influenced climates.

## 5. Correlation & Relationship Analysis

In [ ]:
numeric_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Ethiopia')
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(df['T2M'], df['RH2M'], alpha=0.3, color='teal', s=5)
ax1.set_title('T2M vs RH2M')
ax1.set_xlabel('T2M (°C)')
ax1.set_ylabel('RH2M (%)')

ax2.scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.3, color='coral', s=5)
ax2.set_title('T2M_RANGE vs WS2M')
ax2.set_xlabel('T2M_RANGE (°C)')
ax2.set_ylabel('WS2M (m/s)')

plt.tight_layout()
plt.show()

**Three Strongest Correlations:**
1. **T2M & T2M_MAX / T2M_MIN** — Very strong positive correlation (expected, as max and min temperatures track mean temperature).
2. **T2M & QV2M** — Strong positive correlation: warmer air holds more moisture, increasing specific humidity.
3. **T2M & RH2M** — Moderate negative correlation: as temperature rises, relative humidity tends to drop unless moisture increases proportionally.

## 6. Distribution Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
rain_nonzero = df['PRECTOTCORR'][df['PRECTOTCORR'] > 0]
ax.hist(np.log1p(rain_nonzero), bins=50, color='steelblue', edgecolor='white')
ax.set_title('Distribution of PRECTOTCORR (log scale) — Ethiopia')
ax.set_xlabel('log(1 + PRECTOTCORR)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

**Precipitation Distribution:**
The raw PRECTOTCORR distribution is heavily right-skewed — most days have zero or very low rainfall with occasional extreme events. After applying log transformation, the distribution becomes more bell-shaped, confirming a log-normal pattern typical of daily precipitation data.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['T2M'], df['RH2M'], s=df['PRECTOTCORR'] * 3 + 1,
            alpha=0.3, color='mediumseagreen', edgecolors='none')
plt.title('Bubble Chart: T2M vs RH2M (bubble size = PRECTOTCORR) — Ethiopia')
plt.xlabel('T2M (°C)')
plt.ylabel('RH2M (%)')
plt.tight_layout()
plt.show()